# Câu a và b: lý thuyết → áp dụng dữ liệu

Mỗi câu gồm đúng hai phần: Markdown trình bày các bước và công thức chưa thay số; code bên dưới tính từ dữ liệu rồi in phép thay số và diễn giải.

**Cách chạy:** đặt notebook và `retail_sales_dataset.csv` cùng thư mục, mở notebook từ thư mục đó rồi chọn **Run All**. Câu a dùng xấp xỉ chuẩn như phần giải thích đã thống nhất. Câu b trình bày χ² có điều kiện và bootstrap bổ sung.

## Câu a. Ước lượng kỳ vọng tổng thể của tổng tiền giao dịch

### Phần 1. Các bước lý thuyết (chưa thay số)

**Bước 1. Xác định biến và tham số**

Chọn $X=\text{Total Amount}$, tổng tiền của một giao dịch. Tham số cần ước lượng là $\mu=E(X)$.

**Bước 2. Tính các đặc trưng mẫu và ước lượng điểm**

Với mẫu $x_1,\ldots,x_n$, tính:

$$\bar x=\frac{1}{n}\sum_{i=1}^{n}x_i,$$
$$s^2=\frac{\sum_{i=1}^{n}(x_i-\bar x)^2}{n-1},\qquad s=\sqrt{s^2}.$$

Ước lượng điểm của kỳ vọng tổng thể là $\hat\mu=\bar x$.

**Bước 3. Chọn phương pháp và nêu điều kiện áp dụng**

Giả sử các quan sát độc lập, cùng phân phối, có tính đại diện và phương sai tổng thể hữu hạn. Khi mẫu đủ lớn, theo định lý giới hạn trung tâm và định lý Slutsky:

$$\frac{\bar X-\mu}{S/\sqrt n}\approx N(0,1).$$

Do đó dùng khoảng tin cậy **xấp xỉ chuẩn**, thay độ lệch chuẩn tổng thể chưa biết bằng độ lệch chuẩn mẫu. Mẫu lớn làm phân phối của **trung bình mẫu** xấp xỉ chuẩn, không làm các giá trị giao dịch tự trở thành phân phối chuẩn. Tính độc lập và đại diện là giả định, không thể xác nhận chỉ từ tệp dữ liệu.

**Bước 4. Xác định phân vị và sai số chuẩn**

Với mức tin cậy $1-\alpha$, đặt $z_{1-\alpha/2}=\Phi^{-1}(1-\alpha/2)$, trong đó $\Phi$ là hàm phân phối chuẩn tắc.

$$SE=\frac{s}{\sqrt n},\qquad E=z_{1-\alpha/2}\,SE.$$

**Bước 5. Xây dựng khoảng tin cậy**

$$CI(\mu)=\left[\bar x-z_{1-\alpha/2}\frac{s}{\sqrt n};\ \bar x+z_{1-\alpha/2}\frac{s}{\sqrt n}\right].$$

**Bước 6. Diễn giải theo ngữ cảnh**

Dùng hai cận đã tính để diễn giải khoảng ước lượng cho tổng tiền trung bình của một giao dịch trong tổng thể mục tiêu, dưới các giả định đã nêu. Đây không phải khoảng chứa giá trị của các giao dịch riêng lẻ. Đơn vị tiền tệ chưa xác định trong tệp.

### Phần 2. Code áp dụng dữ liệu và thay số

Code bên dưới tính trực tiếp trung bình, phương sai mẫu hiệu chỉnh, sai số chuẩn và hai cận; `norm.ppf` chỉ được dùng để lấy phân vị chuẩn.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm, chi2, normaltest

# Notebook: đặt dữ liệu cùng thư mục và chạy từ thư mục đó.
# Script .py: mặc định tìm dữ liệu trong thư mục chứa script.
BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_PATH = BASE / "retail_sales_dataset.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu: {DATA_PATH}. Đặt CSV cùng thư mục.")
df = pd.read_csv(DATA_PATH)

# Bước 1: lấy biến tổng tiền, kiểm tra dữ liệu (không tự ý loại dòng).
x = pd.to_numeric(df["Total Amount"], errors="raise").to_numpy(dtype=float)
if len(x) < 50 or not np.isfinite(x).all():
    raise ValueError("Cần ít nhất 50 giá trị tổng tiền hợp lệ, không thiếu.")

# Bước 2: tính trực tiếp các đặc trưng mẫu, dùng mẫu số n - 1.
n = len(x)
x_bar = float(x.sum() / n)
sum_squared_deviations = float(((x - x_bar) ** 2).sum())
s2 = sum_squared_deviations / (n - 1)
s = np.sqrt(s2)

# Bước 3-5: mẫu lớn, dùng xấp xỉ chuẩn với sigma được thay bằng s.
confidence = 0.95
alpha = 1 - confidence
z = float(norm.ppf(1 - alpha / 2))
se = s / np.sqrt(n)
margin = z * se
lower_a = x_bar - margin
upper_a = x_bar + margin

# In phép thay số và kết quả, không chỉ in hai cận.
print("CÂU A - KHOẢNG TIN CẬY 95% CHO KỲ VỌNG TỔNG TIỀN")
print("Bước 1: X = Total Amount; tham số cần ước lượng: mu = E(X).")
print(f"Bước 2: n = {n}; tổng x_i = {x.sum():.6f}")
print(f"  x_bar = tổng x_i / n = {x.sum():.6f} / {n} = {x_bar:.6f}")
print(f"  Tổng (x_i - x_bar)^2 = {sum_squared_deviations:.6f}")
print(f"  s² = {sum_squared_deviations:.6f} / ({n} - 1) = {s2:.6f}")
print(f"  s = sqrt({s2:.6f}) = {s:.6f}")
print(f"  Ước lượng điểm mu_hat = x_bar = {x_bar:.6f}")
print("Bước 3: dùng xấp xỉ chuẩn của trung bình mẫu khi n lớn.")
print("  Giả định: mẫu độc lập, cùng phân phối, đại diện; phương sai hữu hạn.")
print(f"Bước 4: alpha = {alpha:.2f}; z_(1-alpha/2) = z_0.975 = {z:.6f}")
print(f"  SE = s / sqrt(n) = {s:.6f} / sqrt({n}) = {se:.6f}")
print(f"  E = z × SE = {z:.6f} × {se:.6f} = {margin:.6f}")
print(f"Bước 5: CI = [{x_bar:.6f} - {z:.6f} × {se:.6f}; "
      f"{x_bar:.6f} + {z:.6f} × {se:.6f}]")
print(f"  Khoảng tin cậy 95% xấp xỉ: [{lower_a:.2f}; {upper_a:.2f}]")
print(f"Bước 6: với độ tin cậy xấp xỉ 95%, tổng tiền trung bình của một giao dịch "
      f"trong tổng thể mục tiêu được ước lượng từ {lower_a:.2f} đến {upper_a:.2f} "
      "đơn vị tiền, dưới các giả định trên.")
print("  Khoảng này dành cho trung bình tổng thể, không phải các giao dịch riêng lẻ.")


CÂU A - KHOẢNG TIN CẬY 95% CHO KỲ VỌNG TỔNG TIỀN
Bước 1: X = Total Amount; tham số cần ước lượng: mu = E(X).
Bước 2: n = 1000; tổng x_i = 456000.000000
  x_bar = tổng x_i / n = 456000.000000 / 1000 = 456.000000
  Tổng (x_i - x_bar)^2 = 313283750.000000
  s² = 313283750.000000 / (1000 - 1) = 313597.347347
  s = sqrt(313597.347347) = 559.997632
  Ước lượng điểm mu_hat = x_bar = 456.000000
Bước 3: dùng xấp xỉ chuẩn của trung bình mẫu khi n lớn.
  Giả định: mẫu độc lập, cùng phân phối, đại diện; phương sai hữu hạn.
Bước 4: alpha = 0.05; z_(1-alpha/2) = z_0.975 = 1.959964
  SE = s / sqrt(n) = 559.997632 / sqrt(1000) = 17.708680
  E = z × SE = 1.959964 × 17.708680 = 34.708375
Bước 5: CI = [456.000000 - 1.959964 × 17.708680; 456.000000 + 1.959964 × 17.708680]
  Khoảng tin cậy 95% xấp xỉ: [421.29; 490.71]
Bước 6: với độ tin cậy xấp xỉ 95%, tổng tiền trung bình của một giao dịch trong tổng thể mục tiêu được ước lượng từ 421.29 đến 490.71 đơn vị tiền, dưới các giả định trên.
  Khoảng này dành ch

## Câu b. Ước lượng phương sai tổng thể của tuổi

### Phần 1. Các bước lý thuyết (chưa thay số)

**Bước 1. Xác định biến và tham số**

Chọn $X=\text{Age}$, tuổi khách hàng của giao dịch. Tham số cần ước lượng là $\sigma^2=\operatorname{Var}(X)$, có đơn vị năm².

**Bước 2. Tính phương sai mẫu và ước lượng điểm**

$$\bar x=\frac{1}{n}\sum_{i=1}^{n}x_i,\qquad s^2=\frac{\sum_{i=1}^{n}(x_i-\bar x)^2}{n-1}.$$

Ước lượng điểm là $\hat\sigma^2=s^2$.

**Bước 3. Nêu điều kiện của phương pháp $\chi^2$**

Nếu mẫu được lấy độc lập từ tổng thể có **phân phối chuẩn**, thì:

$$\frac{(n-1)S^2}{\sigma^2}\sim\chi^2_{n-1}.$$

Đây là điều kiện để khoảng sau có độ phủ chính xác. Mẫu lớn không tự thay thế điều kiện chuẩn. Tính đại diện cũng cần thiết để suy rộng theo ngữ cảnh.

**Bước 4. Xác định bậc tự do và phân vị**

Với mức tin cậy $1-\alpha$, đặt $\nu=n-1$, $q_L=\chi^2_{\alpha/2;\nu}$ và $q_U=\chi^2_{1-\alpha/2;\nu}$. Ký hiệu $\chi^2_{q;\nu}$ là phân vị có xác suất tích lũy bên trái bằng $q$.

**Bước 5. Biến đổi bất đẳng thức và tính khoảng**

$$P\left(q_L\leq\frac{(n-1)S^2}{\sigma^2}\leq q_U\right)=1-\alpha.$$

Vì các đại lượng đều dương, suy ra:

$$CI(\sigma^2)=\left[\frac{(n-1)s^2}{q_U};\ \frac{(n-1)s^2}{q_L}\right].$$

Cận dưới chia cho phân vị lớn; cận trên chia cho phân vị nhỏ.

**Bước 6. Kiểm tra điều kiện và diễn giải**

Code dùng kiểm định D’Agostino-Pearson như một chẩn đoán phụ:

- $H_0$: tuổi có phân phối chuẩn.
- $H_1$: tuổi không có phân phối chuẩn.
- Nếu p-value nhỏ hơn mức ý nghĩa, bác bỏ $H_0$; không báo cáo khoảng $\chi^2$ như một khoảng có độ phủ được bảo đảm cho dữ liệu này.
- Nếu không bác bỏ $H_0$, điều đó vẫn không chứng minh tổng thể chuẩn.

Khoảng $\chi^2$ được diễn giải **có điều kiện theo giả định tổng thể chuẩn**. Đơn vị hai cận là năm².

**Bổ sung khi giả định chuẩn không được dữ liệu ủng hộ: bootstrap percentile**

1. Lấy có hoàn lại một mẫu gồm $n$ tuổi từ dữ liệu ban đầu.
2. Tính phương sai mẫu hiệu chỉnh $s_b^{2*}$ của mẫu đó.
3. Lặp lại $B$ lần để thu được $s_1^{2*},\ldots,s_B^{2*}$.
4. Lấy khoảng:

$$CI_{\mathrm{boot}}(\sigma^2)=\left[Q_{\alpha/2}(s_1^{2*},\ldots,s_B^{2*});\ Q_{1-\alpha/2}(s_1^{2*},\ldots,s_B^{2*})\right].$$

Bootstrap không yêu cầu tổng thể chuẩn nhưng cần mẫu độc lập, cùng phân phối, đại diện, mômen bậc bốn hữu hạn và mẫu đủ lớn. Khoảng có độ phủ xấp xỉ, kèm sai số mô phỏng.

### Phần 2. Code áp dụng dữ liệu và thay số

Chạy code câu a trước để nạp dữ liệu và thư viện. `chi2.ppf` chỉ lấy phân vị; code tự tính hai cận. `normaltest` kiểm tra giả định, không tạo khoảng tin cậy. Bootstrap được lập trình trực tiếp bằng lấy mẫu có hoàn lại; `np.quantile` chỉ lấy phân vị thực nghiệm.


In [2]:
# Bước 1: chọn biến tuổi.
age = pd.to_numeric(df["Age"], errors="raise").to_numpy(dtype=float)
if len(age) < 50 or not np.isfinite(age).all():
    raise ValueError("Cần ít nhất 50 giá trị tuổi hợp lệ, không thiếu.")

# Bước 2: tính đặc trưng mẫu bằng công thức.
n_age = len(age)
age_mean = float(age.sum() / n_age)
age_ss = float(((age - age_mean) ** 2).sum())
age_s2 = age_ss / (n_age - 1)

# Bước 3-5: khoảng chi-square, có điều kiện tổng thể chuẩn.
confidence_b = 0.95
alpha_b = 1 - confidence_b
dof = n_age - 1
q_low = float(chi2.ppf(alpha_b / 2, dof))
q_high = float(chi2.ppf(1 - alpha_b / 2, dof))
numerator = dof * age_s2
lower_b = numerator / q_high
upper_b = numerator / q_low

print("CÂU B - KHOẢNG TIN CẬY 95% CHO PHƯƠNG SAI TUỔI")
print("Bước 1: X = Age; tham số cần ước lượng: sigma² = Var(X).")
print(f"Bước 2: n = {n_age}; tổng tuổi = {age.sum():.6f}")
print(f"  x_bar = {age.sum():.6f} / {n_age} = {age_mean:.6f} năm")
print(f"  Tổng (x_i - x_bar)^2 = {age_ss:.6f}")
print(f"  s² = {age_ss:.6f} / ({n_age} - 1) = {age_s2:.6f} năm²")
print(f"  Ước lượng điểm sigma_hat² = s² = {age_s2:.6f} năm²")
print("Bước 3: công thức chi-square yêu cầu mẫu độc lập từ tổng thể chuẩn.")
print(f"Bước 4: alpha = {alpha_b:.2f}; bậc tự do = {n_age} - 1 = {dof}")
print(f"  q_L = chi²_(0.025; {dof}) = {q_low:.6f}")
print(f"  q_U = chi²_(0.975; {dof}) = {q_high:.6f}")
print(f"Bước 5: (n - 1)s² = {dof} × {age_s2:.6f} = {numerator:.6f}")
print(f"  Cận dưới = {numerator:.6f} / {q_high:.6f} = {lower_b:.6f}")
print(f"  Cận trên = {numerator:.6f} / {q_low:.6f} = {upper_b:.6f}")
print(f"  Khoảng chi-square CÓ ĐIỀU KIỆN: [{lower_b:.2f}; {upper_b:.2f}] năm²")

# Bước 6: kiểm tra phụ giả định chuẩn và diễn giải có điều kiện.
test = normaltest(age)
normality_alpha = 0.05
print(f"Bước 6: kiểm định chuẩn D'Agostino-Pearson: K² = {test.statistic:.6f}; "
      f"p-value = {test.pvalue:.3e}")
if test.pvalue < normality_alpha:
    print("  Bác bỏ giả thuyết chuẩn ở mức ý nghĩa 5%.")
    print("  Không coi khoảng chi-square trên là khoảng được bảo đảm độ phủ 95%.")
else:
    print("  Chưa bác bỏ giả thuyết chuẩn ở mức 5%; không chứng minh tổng thể chuẩn.")
print(f"  Nếu tổng thể tuổi chuẩn và mẫu độc lập, đại diện: phương sai tuổi được "
      f"ước lượng từ {lower_b:.2f} đến {upper_b:.2f} năm² với độ tin cậy 95%.")

# Bổ sung: bootstrap phi tham số, không cần giả định tổng thể chuẩn.
B = 20000
seed = 20261011
rng = np.random.default_rng(seed)
bootstrap_variances = np.empty(B)
for start in range(0, B, 250):
    batch_size = min(250, B - start)
    # Mỗi hàng là một mẫu n tuổi, lấy có hoàn lại.
    samples = age[rng.integers(0, n_age, size=(batch_size, n_age))]
    sample_means = samples.sum(axis=1) / n_age
    bootstrap_variances[start:start + batch_size] = (
        ((samples - sample_means[:, None]) ** 2).sum(axis=1) / (n_age - 1)
    )
boot_low, boot_high = np.quantile(
    bootstrap_variances, [alpha_b / 2, 1 - alpha_b / 2], method="linear"
)
print("\nBỔ SUNG - KHOẢNG BOOTSTRAP PHƯƠNG SAI TUỔI")
print(f"  B = {B}; seed = {seed}; mỗi lần lấy có hoàn lại {n_age} tuổi.")
print("  Mỗi phương sai bootstrap dùng tổng bình phương độ lệch / (n - 1).")
print(f"  Q_0.025(s²*) = {boot_low:.6f}; Q_0.975(s²*) = {boot_high:.6f}")
print(f"  Khoảng bootstrap 95% xấp xỉ: [{boot_low:.2f}; {boot_high:.2f}] năm²")
print(f"  Dưới giả định mẫu độc lập, cùng phân phối, đại diện, mômen bậc bốn hữu hạn: "
      f"phương sai tuổi tổng thể được ước lượng từ {boot_low:.2f} đến {boot_high:.2f} năm².")
print("  Đây là khoảng xấp xỉ và có sai số mô phỏng.")


CÂU B - KHOẢNG TIN CẬY 95% CHO PHƯƠNG SAI TUỔI
Bước 1: X = Age; tham số cần ước lượng: sigma² = Var(X).
Bước 2: n = 1000; tổng tuổi = 41392.000000
  x_bar = 41392.000000 / 1000 = 41.392000 năm
  Tổng (x_i - x_bar)^2 = 186994.336000
  s² = 186994.336000 / (1000 - 1) = 187.181518 năm²
  Ước lượng điểm sigma_hat² = s² = 187.181518 năm²
Bước 3: công thức chi-square yêu cầu mẫu độc lập từ tổng thể chuẩn.
Bước 4: alpha = 0.05; bậc tự do = 1000 - 1 = 999
  q_L = chi²_(0.025; 999) = 913.300998
  q_U = chi²_(0.975; 999) = 1088.487068
Bước 5: (n - 1)s² = 999 × 187.181518 = 186994.336000
  Cận dưới = 186994.336000 / 1088.487068 = 171.792887
  Cận trên = 186994.336000 / 913.300998 = 204.745573
  Khoảng chi-square CÓ ĐIỀU KIỆN: [171.79; 204.75] năm²
Bước 6: kiểm định chuẩn D'Agostino-Pearson: K² = 742.479870; p-value = 5.923e-162
  Bác bỏ giả thuyết chuẩn ở mức ý nghĩa 5%.
  Không coi khoảng chi-square trên là khoảng được bảo đảm độ phủ 95%.
  Nếu tổng thể tuổi chuẩn và mẫu độc lập, đại diện: phươn